# Crime in Boston (2015-2018): 
## Programming for Data Science, final project

## Team members and GitHub link
- Héloïse MOYNOT (team head)
- Estelle HU (team member)
- Alice QUENTREC (team member)
- Tashaly SOBNACK (team member)

GitHub repository: https://github.com/heloisemoynot-commits/Boston_crime_project

## About the dataset

We use the **Crimes in Boston** dataset published by the Boston Police Department (BPD) and shared on Kaggle
(https://www.kaggle.com/datasets/AnalyzeBoston/crimes-in-boston).
Each row is one offense reported by the police: something they had to respond to, from a robbery to a car accident or a medical call.
One incident (INCIDENT_NUMBER) can have several rows if several offenses were recorded (for example an assault and a robbery at the same time).
The data goes from June 2015 to September 2018.

It is a multidimensional dataset:
- **temporal dimension** : when the incident happened (OCCURRED_ON_DATE, plus YEAR, MONTH, DAY_OF_WEEK, HOUR)
- **spatial dimension** : where it happened (Lat, Long, DISTRICT, STREET, REPORTING_AREA)
- **analytical dimension**: what happened (OFFENSE_CODE_GROUP, OFFENSE_DESCRIPTION, UCR_PART for the seriousness, SHOOTING)

The zip also has offense_codes.csv (code -> name), but we don't use it because crime.csv already has the offense description in text.

Our idea is to answer simple questions a city or the police could ask: what are the most common incidents and where, which
things tend to happen together, how many incidents to expect in the next months, and where the hotspots are.

In [2]:
# Import libraries
import os
import logging
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from mlxtend.frequent_patterns import apriori, association_rules   
from prophet import Prophet                                         
from sklearn.cluster import DBSCAN                                  
from sklearn.metrics import mean_absolute_error

from dash import Dash, html, dcc                                    
from IPython.display import display                                 

warnings.simplefilter("ignore")
logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").setLevel(logging.WARNING)

c:\Users\heloi\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
c:\Users\heloi\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


## Step 1. Loading and exploring the data

In [3]:
# Function for reading the crime CSV file
# The input is the path to the csv file, the file encoding
# The output is a dataframe with the raw data
def load_data(file_path, encoding="latin-1"):

    # the Kaggle file is not utf-8, so we read it as latin-1 
    df = pd.read_csv(file_path, encoding=encoding, low_memory=False)
    
    # some text cells only contain spaces so we turn them into NaN
    text_cols = df.select_dtypes(include="object").columns.union(df.select_dtypes(include="string").columns)
    df[text_cols] = df[text_cols].replace(r"^\s*$", np.nan, regex=True)
    print(f"Loaded {file_path}: {df.shape[0]} rows, {df.shape[1]} columns")
    return df

In [4]:
# Function to summarize the dataset
# The input is a dataframe
# The output is a summary dataframe (type, number of missing values, % missing, number of unique values, min, max for each column)
def summarize_data(df):
    print("Number of rows:", df.shape[0])
    print("Number of columns:", df.shape[1])
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(),
    })
    # range of values: min and max (works for numbers and also for text, sorted alphabetically)
    summary["min"] = [df[c].dropna().min() if df[c].notna().any() else None for c in df.columns]
    summary["max"] = [df[c].dropna().max() if df[c].notna().any() else None for c in df.columns]
    return summary

###  Our cleaning decisions
After looking at the summary we make these choices (done in `clean_data` below):
- OCCURRED_ON_DATE is read as text, so we convert it to a real datetime.
- Location is just (Lat, Long) as a string, so we **drop it** (duplicate information).
- Some coordinates are missing and some are set to -1 (fake value, it's not in Boston). We set them to NaN and only
  remove those rows for the spatial analysis, so we don't lose incidents for the other queries.
- SHOOTING is Y or empty, so we turn it into 0/1.
- Rows without a DISTRICT are kept but labeled "Unknown".
- We remove exact duplicated rows (same incident, same offense, same everything).
- We keep OFFENSE_CODE, OFFENSE_DESCRIPTION, REPORTING_AREA and STREET in the data but we don't use them much:
  the offense code and description are too detailed (222 codes), so we work with the 67 OFFENSE_CODE_GROUP instead.
- We add TIME_OF_DAY (Night 0-5h, Morning 6-11h, Afternoon 12-17h, Evening 18-23h) and IS_WEEKEND, which are useful for pattern mining.

In [5]:
# Function to clean the dataset and add a few helper columns
# The input is the raw dataframe
# The output is a cleaned copy of the dataframe

def clean_data(df):
    df = df.copy()
    df["OCCURRED_ON_DATE"] = pd.to_datetime(df["OCCURRED_ON_DATE"], errors="coerce")  # text -> datetime
    df = df.dropna(subset=["OCCURRED_ON_DATE"])                                        # an incident without a date is useless here
    df = df.drop(columns=["Location"], errors="ignore")                                # same info as Lat / Long
    df = df.drop_duplicates()

    # coordinates: -1 or anything outside the Boston area -> missing
    bad_coords = ~df["Lat"].between(42.2, 42.45) | ~df["Long"].between(-71.2, -70.95)
    df.loc[bad_coords, ["Lat", "Long"]] = np.nan

    df["SHOOTING"] = df["SHOOTING"].notna().astype(int)        # 'Y' becomes 1, empty becomes 0
    df["DISTRICT"] = df["DISTRICT"].fillna("Unknown")
    df["OFFENSE_CODE_GROUP"] = df["OFFENSE_CODE_GROUP"].fillna("Unknown")

    # new columns used later
    df["TIME_OF_DAY"] = pd.cut(df["HOUR"], bins=[-1, 5, 11, 17, 23],
                               labels=["Night", "Morning", "Afternoon", "Evening"])
    df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin(["Saturday", "Sunday"])
    print(f"After cleaning: {df.shape[0]} rows, {df['Lat'].isna().sum()} rows without valid coordinates")
    return df

## Step 2. Information building

### Query 1 (grouping): which offenses are the most frequent, and in which districts?

**Indicator :** number of reported offenses per offense group (*OFFENSE_CODE_GROUP*), and for the top offenses, how they are split by police district.

**How it's calculated :** we group the rows by *OFFENSE_CODE_GROUP* and count them (*groupby().size()*), keep the top N,
then do a second group by (*OFFENSE_CODE_GROUP*, *DISTRICT*) for those offenses.

**How to read it :** a long bar means this type of incident takes a lot of police time. The district chart shows whether an
offense is concentrated in a few districts (local problem) or spread everywhere (city-wide problem).

In [6]:
# Function to count incidents per offense group and per district
# The input is the cleaned dataframe and the number of offense groups to keep
# The output is a tuple containing : table of counts per offense, figure of the top offenses, figure of top offenses by district

def query_top_offenses(df, top_n=10):
    counts = (df.groupby("OFFENSE_CODE_GROUP").size()
                .sort_values(ascending=False)
                .reset_index(name="incidents"))
    top = counts.head(top_n)

    fig_top = px.bar(top.sort_values("incidents"), x="incidents", y="OFFENSE_CODE_GROUP", orientation="h",
                     title=f"Top {top_n} offense groups (number of reported offenses)",
                     labels={"OFFENSE_CODE_GROUP": "", "incidents": "Number of reported offenses"})

    # same top offenses, split by district
    by_district = (df[df["OFFENSE_CODE_GROUP"].isin(top["OFFENSE_CODE_GROUP"].head(5))]
                     .groupby(["DISTRICT", "OFFENSE_CODE_GROUP"]).size().reset_index(name="incidents"))
    fig_district = px.bar(by_district, x="DISTRICT", y="incidents", color="OFFENSE_CODE_GROUP",
                          title="Top 5 offenses by district", labels={"incidents": "Number of reported offenses"})
    fig_district.update_xaxes(categoryorder="total descending")
    return counts, fig_top, fig_district

### Query 2 (frequent pattern mining): which characteristics tend to appear together?

**Indicator :** association rules A -> B with their *support*, *confidence* and *lift*.

**How it's calculated :** each incident becomes a "transaction" made of items like *offense=Larceny*, *district=D4*,
*time=Evening*, *day=Weekend*. We one-hot encode them and run the Apriori algorithm to find frequent itemsets, then we generate association rules.

- support = share of the transactions that contain A and B
- confidence = P(B | A), i.e. among incidents with A, the share that also have B
- lift = confidence / P(B). A lift above 1 means A and B appear together more often than by chance.

To keep things readable we only keep the 15 most frequent offense groups (which is about 76% of the rows), rules with lift > 1.2, and rules with only one item on the right side. We also remove *redundant rules*: if *{D4, Afternoon} -> Larceny* already
has a lift of 2.6, the longer rule *{D4, Afternoon, Weekday} -> Larceny* with a lift of 2.5 doesn't add anything, so a longer
rule is only kept if its lift is higher than all its shorter versions.

We don't use *UCR_PART* as an item: it is directly given by the offense type, so it only produces obvious rules with a huge confidence that teach us nothing.

**How to read it :** a rule like *{offense=Larceny} -> {district=D4}* with lift 2 means larcenies are twice as likely to
be in district D4 than an average incident. This can help to decide where/when to send patrols for a given offense.

In [7]:
# Function to build the "transactions" for pattern mining
# The input is the cleaned dataframe and the number of offense groups to keep
# The output is a one-hot (True/False) dataframe, one column per item

def build_transactions(df, top_offenses=15):
    keep = df["OFFENSE_CODE_GROUP"].value_counts().head(top_offenses).index   # because rare offenses would never be frequent anyway
    sub = df[df["OFFENSE_CODE_GROUP"].isin(keep)]
    items = pd.DataFrame({
        "offense": "offense=" + sub["OFFENSE_CODE_GROUP"].astype(str),
        "district": "district=" + sub["DISTRICT"].astype(str),
        "time": "time=" + sub["TIME_OF_DAY"].astype(str),
        "day": np.where(sub["IS_WEEKEND"], "day=Weekend", "day=Weekday"),
    })
    # one column per possible item, True if the incident has it
    onehot = pd.get_dummies(items, prefix="", prefix_sep="").astype(bool)
    return onehot


# Function to mine frequent itemsets and association rules
# The input is a one-hot dataframe, minimum support, and minimum lift
# The output is a tuple containing (rules dataframe sorted by lift, scatter figure support vs confidence)

def query_patterns(onehot, min_support=0.005, min_lift=1.2):
    itemsets = apriori(onehot, min_support=min_support, use_colnames=True)
    rules = association_rules(itemsets, metric="lift", min_threshold=min_lift)
    rules = rules[rules["consequents"].apply(len) == 1]    # one item on the right side -> easier to read

    # remove redundant rules: a rule is dropped if a shorter rule with the same consequent has a lift at least as high

    # Function to check if one rule is redundant
    # he input is one rule (a row of the rules dataframe)
    # The output is True if a shorter rule with the same consequent is at least as good
    
    def is_redundant(r):
        same = rules[(rules["consequents"] == r["consequents"]) & (rules["antecedents"].apply(lambda a: a < r["antecedents"]))]
        return (same["lift"] >= r["lift"]).any()
    rules = rules[~rules.apply(is_redundant, axis=1)]
    # sets are hard to read, so we turn them into strings
    rules["antecedents"] = rules["antecedents"].apply(lambda s: ", ".join(sorted(s)))
    rules["consequents"] = rules["consequents"].apply(lambda s: ", ".join(sorted(s)))
    rules = rules[["antecedents", "consequents", "support", "confidence", "lift"]].sort_values("lift", ascending=False)
    print(f"{len(itemsets)} frequent itemsets, {len(rules)} non-redundant rules with lift >= {min_lift}")

    fig = px.scatter(rules, x="support", y="confidence", color="lift", size="lift",
                     hover_data=["antecedents", "consequents"],
                     title="Association rules (hover to see the rule)", color_continuous_scale="Viridis")
    return rules.reset_index(drop=True), fig

### Query 3 (temporal): how many incidents should we expect in the coming months?

**Indicator :** number of incidents per day, and a forecast of this number for the next 6 months.

**How it's calculated :** we count the number of different incidents (*INCIDENT_NUMBER*) per day, so an incident with 3 offenses counts once. Then we fit a Prophet model (like in the time series lab), with weekly and yearly seasonality, plus the US public holidays (Christmas, Thanksgiving... are much quieter days). The first and last months are cut because they are incomplete (the data starts and ends in the middle of a month).

To check the model, we first keep the last 90 days aside, train on the rest and compute the *MAE* (mean absolute error,
in incidents per day) on these 90 days. Then we retrain on everything and forecast 180 days.

**How to read it :** the black dots are the real daily counts, the blue line is the forecast and the shaded area
is the 95% uncertainty interval. The components plot shows the trend, the day of week effect and the season effect
(for example, more incidents in summer). The smaller the MAE compared to the average per day, the more we can trust the forecast.

In [8]:
# Function to build the daily time series of incidents
# The input is the cleaned dataframe
# The output is a dataframe with columns ds (date) and y (number of different incidents), in the Prophet format

def daily_counts(df):
    # nunique: one incident with several offenses is only counted once
    daily = df.set_index("OCCURRED_ON_DATE").resample("D")["INCIDENT_NUMBER"].nunique().reset_index()
    daily.columns = ["ds", "y"]

    # drop the first and last (incomplete) months
    first_full = (daily["ds"].min() + pd.offsets.MonthBegin(1)).normalize()
    last_full = (daily["ds"].max() - pd.offsets.MonthEnd(1)).normalize()
    return daily[(daily["ds"] >= first_full) & (daily["ds"] <= last_full)].reset_index(drop=True)


# Function to create the Prophet model (same settings for the test and the final model)
# The input is nothing
# The output is an untrained Prophet model

def make_prophet():
    model = Prophet(interval_width=0.95, weekly_seasonality=True, yearly_seasonality=True)
    model.add_country_holidays(country_name="US")   # public holidays have fewer incidents
    return model


# Function to evaluate and run the Prophet forecast
# The input is daily series (ds, y), number of days kept for the test, number of days to forecast
# The output is (forecast dataframe, MAE on the test period, forecast figure, components figure)

def query_forecast(daily, test_days=90, horizon=180):
    # 1) evaluation: train without the last test_days, compare with the real values
    train, test = daily.iloc[:-test_days], daily.iloc[-test_days:]
    m_test = make_prophet().fit(train)
    pred = m_test.predict(test[["ds"]])
    mae = mean_absolute_error(test["y"], pred["yhat"])
    print(f"MAE on the last {test_days} days: {mae:.1f} incidents/day (mean = {test['y'].mean():.1f}/day)")

    # 2) final model on all the data
    model = make_prophet().fit(daily)
    future = model.make_future_dataframe(periods=horizon)
    forecast = model.predict(future)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat_upper"], line=dict(width=0), showlegend=False, name="upper bound"))
    fig.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat_lower"], line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(0,114,178,0.2)", name="95% interval"))
    fig.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat"], line=dict(color="#0072B2"), name="Forecast"))
    fig.add_trace(go.Scatter(x=daily["ds"], y=daily["y"], mode="markers", marker=dict(size=3, color="black"),
                             name="Observed"))
    fig.update_layout(title=f"Daily incidents and {horizon}-day forecast (Prophet, test MAE = {mae:.1f})",
                      xaxis_title="Date", yaxis_title="Incidents per day")

    from prophet.plot import plot_components_plotly
    fig_comp = plot_components_plotly(model, forecast)
    return forecast, mae, fig, fig_comp

### Query 4 (spatial): where are the crime hotspots?

**Indicator :** clusters of incidents (hotspots) for a chosen offense, with their size and center.

**How it's calculated :** we take the incidents with valid coordinates for one offense group (by default *Larceny*)
and run DBSCAN with the haversine distance (so distances are real distances on the Earth, like OPTICS in the spatial lab).
- *eps_m* = radius in meters (here 100 m, about one block)
- *min_samples* = minimum number of incidents in this radius to start a cluster (here 80)

We tried a few values: with 150 m and 30 points, almost all of downtown and Back Bay merged into one giant cluster of 11,000
points, which is not a "hotspot" anymore. With 100 m and 80 points we get about 30 small, readable hotspots.

Points that don't belong to any dense area get the label -1 (noise) and are not shown as hotspots.
If there are too many points we take a random sample to keep it fast (DBSCAN memory grows quickly).

**How to read it :** each color on the map is a hotspot, a small zone where this offense happens much more than
around it. The table gives, for each hotspot, the number of incidents and the most common street. The share of points in
clusters tells if the offense is concentrated (high share) or spread out (low share).

In [9]:
# Function to find spatial hotspots with DBSCAN
# The input is a cleaned dataframe, offense group (None = all), radius in meters, min points, max number of points used
# The output is (points with their cluster label, summary table per cluster, map figure)

def query_hotspots(df, offense="Larceny", eps_m=100, min_samples=80, max_points=30000):
    pts = df.dropna(subset=["Lat", "Long"])
    if offense is not None:
        pts = pts[pts["OFFENSE_CODE_GROUP"] == offense]
    if len(pts) > max_points:
        pts = pts.sample(max_points, random_state=42)   # it is fixed seed so the result is reproducible
    pts = pts.copy()

    earth_radius_m = 6371000
    coords = np.radians(pts[["Lat", "Long"]].values)    # haversine needs [lat, lon] in radians
    db = DBSCAN(eps=eps_m / earth_radius_m, min_samples=min_samples, metric="haversine", algorithm="ball_tree")
    pts["cluster"] = db.fit_predict(coords)

    clustered = pts[pts["cluster"] != -1]
    summary = (clustered.groupby("cluster")
                 .agg(incidents=("cluster", "size"), lat=("Lat", "mean"), lon=("Long", "mean"),
                      main_street=("STREET", lambda s: s.mode().iat[0] if s.notna().any() else "?"),
                      district=("DISTRICT", lambda s: s.mode().iat[0]))
                 .sort_values("incidents", ascending=False).reset_index())
    share = len(clustered) / len(pts) * 100
    print(f"{offense or 'All offenses'}: {len(pts)} points, {summary.shape[0]} hotspots, {share:.1f}% of points in a hotspot")

    # we draw the map as a simple longitude/latitude scatter plot (no background map to download,
    # so it works everywhere, even offline). The grey "noise" points draw the shape of the city.
    noise = pts[pts["cluster"] == -1]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=noise["Long"], y=noise["Lat"], mode="markers", name="not in a hotspot",
                             marker=dict(size=3, color="lightgrey"), hoverinfo="skip"))
    for c in sorted(clustered["cluster"].unique()):
        g = clustered[clustered["cluster"] == c]
        fig.add_trace(go.Scatter(x=g["Long"], y=g["Lat"], mode="markers", name=f"hotspot {c}",
                                 marker=dict(size=5), text=g["STREET"], showlegend=False,
                                 hovertemplate=f"hotspot {c}<br>%{{text}}<extra></extra>"))
    # label the 5 biggest hotspots with their main street
    offsets = [(-150, -70), (170, -40), (-170, 30), (170, 40), (-120, 90)]   # spread the labels so they don't overlap
    for (_, row), (ax, ay) in zip(summary.head(5).iterrows(), offsets):
        fig.add_annotation(x=row["lon"], y=row["lat"], text=f"{row['main_street']} ({row['incidents']})",
                           showarrow=True, arrowhead=2, ax=ax, ay=ay, bgcolor="white", bordercolor="grey")
    fig.update_layout(title=f"Hotspots of {offense or 'all offenses'} (DBSCAN, eps = {eps_m} m, {summary.shape[0]} hotspots)",
                      xaxis_title="Longitude", yaxis_title="Latitude", height=650, plot_bgcolor="white")
    # 1 degree of longitude is shorter than 1 degree of latitude at Boston (x cos 42°), so we fix the ratio
    fig.update_yaxes(scaleanchor="x", scaleratio=1 / np.cos(np.radians(42.33)))
    return pts, summary, fig

### Bonus (external dataset): does the weather change the number of incidents?

**Indicator :** correlation between the daily maximum temperature and the number of incidents per day, plus the average
number of incidents per temperature range.

**How it's calculated :** we load daily weather at Boston Logan airport from NOAA (station *USW00014739*, columns *DATE*, *TMAX*, *PRCP*),
join it with the daily counts on the date, compute the Pearson correlation and plot incidents vs temperature.

**How to read it :** a positive correlation means there are more incidents on hot days. The correlation does not prove
that heat causes crime ( possibly because in summer people are outside more, school is off, ...), it just helps to interpret the yearly seasonality we saw in query 3.

In [10]:
# Function to load the NOAA weather file and merge it with the daily counts
# The input is the path to the NOAA csv and the daily counts (ds, y)
# # The output is (merged dataframe, correlation value, scatter figure, bar figure) or None if the file is missing

def query_weather(weather_path, daily):
    if not os.path.exists(weather_path):
        print(f"No weather file found at {weather_path}, bonus skipped")
        return None
    w = pd.read_csv(weather_path)
    w["ds"] = pd.to_datetime(w["DATE"])
    merged = daily.merge(w[["ds", "TMAX", "PRCP"]], on="ds", how="inner").dropna(subset=["TMAX"])
    corr = merged["y"].corr(merged["TMAX"])
    print(f"Correlation between max temperature and daily incidents: {corr:.2f}")

    fig = px.scatter(merged, x="TMAX", y="y", opacity=0.5,
                labels={"TMAX": "Max temperature (°C)", "y": "Incidents per day"},
                title=f"Daily incidents vs temperature (r = {corr:.2f})")
    
    # trend line computed with numpy (linear regression y = a*x + b), so we don't need statsmodels
    a, b = np.polyfit(merged["TMAX"], merged["y"], 1)
    x_line = np.array([merged["TMAX"].min(), merged["TMAX"].max()])
    fig.add_trace(go.Scatter(x=x_line, y=a * x_line + b, mode="lines", name=f"trend (+{a:.1f} incidents per °C)"))

    # average number of incidents for each 5°C range
    merged["temp_range"] = pd.cut(merged["TMAX"], bins=range(-20, 45, 5)).astype(str)
    by_range = merged.groupby("temp_range", sort=False)["y"].agg(["mean", "size"]).reset_index()
    by_range = by_range[by_range["size"] >= 10]           # ranges with less than 10 days are not reliable
    by_range["order"] = by_range["temp_range"].str.extract(r"\((-?\d+)").astype(int)
    fig_range = px.bar(by_range.sort_values("order"), x="temp_range", y="mean",
                       labels={"temp_range": "Max temperature range (°C)", "mean": "Average incidents per day"},
                       title="Average incidents per day by temperature range")
    return merged, corr, fig, fig_range

## Step 3. Dashboard

The dashboard is made with **Dash**. It shows the team, the dataset name and one tab per query.
Since a Dash app only lives while the notebook runs, we also save all the figures in one static HTML file
(`MOYNOT_dashboard.html`) as evidence that it works.

In [11]:
# Function to build the Dash app
# he input is a dict {tab title: list of figures}, team members, dataset name
# The output is the Dash app (not started)

def build_dashboard(sections, team, dataset_name):
    app = Dash(__name__)
    tabs = [dcc.Tab(label=title, children=[dcc.Graph(figure=f) for f in figs])
            for title, figs in sections.items()]
    app.layout = html.Div(style={"fontFamily": "Arial", "margin": "20px"}, children=[
        html.H1("Crime in Boston (2015-2018)"),
        html.P(f"Dataset: {dataset_name}"),
        html.P("Team: " + ", ".join(team)),
        dcc.Tabs(tabs),
    ])
    return app


# Function to save every figure of the dashboard in one static HTML file
# The input is the same dict of sections, team, dataset name, and output path
# The output is the path of the HTML file

def export_dashboard_html(sections, team, dataset_name, path):
    parts = [f"<h1>Crime in Boston (2015-2018)</h1><p>Dataset: {dataset_name}</p><p>Team: {', '.join(team)}</p>"]
    for title, figs in sections.items():
        parts.append(f"<h2>{title}</h2>")
        for f in figs:
            parts.append(pio.to_html(f, full_html=False, include_plotlyjs=False)) 
    from plotly.offline import get_plotlyjs
    plotly_js = "<script type='text/javascript'>" + get_plotlyjs() + "</script>"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("<!DOCTYPE html><html><head><meta charset='utf-8'><title>Boston crime dashboard</title>"
                 + plotly_js + "</head><body style='font-family:Arial;margin:20px'>"
                 + "\n".join(parts) + "</body></html>")
    print("Dashboard saved to", path)
    return path

## Main

In [12]:
# Main block
if __name__ == "__main__":
    # parameters
    CRIME_PATH = "crime.csv"
    WEATHER_PATH = "boston_weather.csv"          # NOAA daily summaries, optional (bonus)
    TEAM = ["Héloïse MOYNOT (team head)", "Estelle HU", "Alice QUENTREC", "Tashaly SOBNACK"]
    DATASET_NAME = "Crimes in Boston (Boston Police Department, Kaggle)"
    DASHBOARD_FILE = "MOYNOT_dashboard.html"
    RUN_DASH = True

    # Step 1: data collection and exploration
    raw = load_data(CRIME_PATH)
    display(raw.head())
    display(summarize_data(raw))
    crimes = clean_data(raw)

    # Step 2: indicators construction
    # Query 1: grouping
    counts, fig_top, fig_district = query_top_offenses(crimes, top_n=10)
    display(counts.head(10))
    fig_top.show()
    fig_district.show()

    # Query 2: frequent patterns
    onehot = build_transactions(crimes, top_offenses=15)
    rules, fig_rules = query_patterns(onehot, min_support=0.005, min_lift=1.2)
    display(rules.head(15))
    fig_rules.show()

    # Query 3: forecasting
    daily = daily_counts(crimes)
    forecast, mae, fig_forecast, fig_components = query_forecast(daily, test_days=90, horizon=180)
    fig_forecast.show()
    fig_components.show()

    # Query 4: spatial clustering
    points, hotspots, fig_map = query_hotspots(crimes, offense="Larceny", eps_m=100, min_samples=80)
    display(hotspots.head(10))
    fig_map.show()

    # Bonus: external weather dataset
    weather = query_weather(WEATHER_PATH, daily)
    if weather is not None:
        fig_weather, fig_weather_range = weather[2], weather[3]
        fig_weather.show()
        fig_weather_range.show()

    # Step 3: dashboard
    sections = {
        "1. Most frequent offenses": [fig_top, fig_district],
        "2. Frequent patterns": [fig_rules],
        "3. Forecast": [fig_forecast, fig_components],
        "4. Hotspots": [fig_map],
    }
    if weather is not None:
        sections["Bonus: weather"] = [fig_weather, fig_weather_range]

    export_dashboard_html(sections, TEAM, DATASET_NAME, DASHBOARD_FILE)
    if RUN_DASH:
        app = build_dashboard(sections, TEAM, DATASET_NAME)
        app.run(jupyter_mode="inline", port=8050)   # use jupyter_mode="external" to open it in a browser tab

Loaded crime.csv: 319073 rows, 17 columns


,INCIDENT_NUMBER,OFFENSE_CODE,OFFENSE_CODE_GROUP,OFFENSE_DESCRIPTION,DISTRICT,REPORTING_AREA,SHOOTING,OCCURRED_ON_DATE,YEAR,MONTH,DAY_OF_WEEK,HOUR,UCR_PART,STREET,Lat,Long,Location
0,I182070945,619,Larceny,LARCENY ALL OTHERS,D14,808,NaN,2018-09-02 13:00:00,2018,9,Sunday,13,Part One,LINCOLN ST,42.357791,-71.139371,"(42.35779134, -71.13937053)"
1,I182070943,1402,Vandalism,VANDALISM,C11,347,NaN,2018-08-21 00:00:00,2018,8,Tuesday,0,Part Two,HECLA ST,42.306821,-71.060300,"(42.30682138, -71.06030035)"
2,I182070941,3410,Towed,TOWED MOTOR VEHICLE,D4,151,NaN,2018-09-03 19:27:00,2018,9,Monday,19,Part Three,CAZENOVE ST,42.346589,-71.072429,"(42.34658879, -71.07242943)"
3,I182070940,3114,Investigate Property,INVESTIGATE PROPERTY,D4,272,NaN,2018-09-03 21:16:00,2018,9,Monday,21,Part Three,NEWCOMB ST,42.334182,-71.078664,"(42.33418175, -71.07866441)"
4,I182070938,3114,Investigate Property,INVESTIGATE PROPERTY,B3,421,NaN,2018-09-03 21:05:00,2018,9,Monday,21,Part Three,DELHI ST,42.275365,-71.090361,"(42.27536542, -71.09036101)"


Number of rows: 319073
Number of columns: 17


,dtype,missing,missing_%,unique_values,min,max
INCIDENT_NUMBER,str,0,0.00,282517,142052550,I182070945
OFFENSE_CODE,int64,0,0.00,222,111,3831
OFFENSE_CODE_GROUP,str,0,0.00,67,Aggravated Assault,Warrant Arrests
OFFENSE_DESCRIPTION,str,0,0.00,244,"A&B HANDS, FEET, ETC. - MED. ATTENTION REQ.",WEAPON - OTHER - OTHER VIOLATION
DISTRICT,str,1765,0.55,12,A1,E5
REPORTING_AREA,str,20250,6.35,878,000,99
SHOOTING,str,318054,99.68,1,Y,Y
OCCURRED_ON_DATE,str,0,0.00,233229,2015-06-15 00:00:00,2018-09-03 21:25:00
YEAR,int64,0,0.00,4,2015,2018
MONTH,int64,0,0.00,12,1,12


After cleaning: 319050 rows, 20743 rows without valid coordinates


,OFFENSE_CODE_GROUP,incidents
0,Motor Vehicle Accident Response,37132
1,Larceny,25935
2,Medical Assistance,23540
3,Investigate Person,18749
4,Other,18073
5,Drug Violation,16545
6,Simple Assault,15826
7,Vandalism,15414
8,Verbal Disputes,13099
9,Towed,11287


... (3 outputs left out)

Larceny: 25102 points, 31 hotspots, 37.4% of points in a hotspot


,cluster,incidents,lat,lon,main_street,district
0,0,2162,42.354758,-71.061155,WASHINGTON ST,A1
1,1,1062,42.350801,-71.075836,BOYLSTON ST,D4
2,2,914,42.348417,-71.079549,HUNTINGTON AVE,D4
3,5,898,42.348674,-71.082937,BOYLSTON ST,D4
4,7,624,42.360025,-71.054918,STATE ST,A1
5,9,397,42.344128,-71.099387,BOYLSTON ST,D4
6,6,262,42.348394,-71.086850,NEWBURY ST,D4
7,15,251,42.334228,-71.076094,HARRISON AVE,D4
8,21,237,42.326966,-71.061986,ALLSTATE RD,C6
9,17,222,42.375542,-71.039595,BORDER ST,A7


Correlation between max temperature and daily incidents: 0.44


Dashboard saved to MOYNOT_dashboard.html


## Results and interpretation

**Exploration :** The file has 319,073 rows and 17 columns, from 15 June 2015 to 3 September 2018 (so 2015 and 2018 are
not complete years). These rows correspond to 282 517 different incidents: about 10% of the incidents have several offenses.
About 6% of the rows have no coordinates (NaN or the fake value -1), 6% have an empty *REPORTING_AREA* (only spaces, that we
turned into NaN), 3% have no street and 0.5% no district. *SHOOTING* is empty 99.7% of the time, which just means
"no shooting" (1,019 offenses with a shooting). We also removed 23 duplicated rows.

**Query 1 :** The most frequent group is *Motor Vehicle Accident Response* (37,132), then *Larceny* (25,935) and
*Medical Assistance* (23,540). So a big part of police work is not "crime" in the strict sense but accidents and medical calls.
B2 (Roxbury), C11 (Dorchester) and D4 (South End / Back Bay) are the busiest districts.

**Query 2 :** The strongest rules (lift between 2 and 2.7) are easy to explain:
- *{district=D4, time=Afternoon} -> {offense=Larceny}* : in D4 in the afternoon, 27.5% of incidents are larcenies, 2.6 times the city average. D4 has the shopping streets (Newbury St, Boylston St, Prudential), so shoplifting makes sense.
- *{offense=Towed} -> {time=Morning}*: 54% of tows happen in the morning (street cleaning, rush-hour parking rules).
- *{day=Weekend, offense=Simple Assault} -> {time=Night}*: assaults on weekend nights, linked to nightlife.
- *{day=Weekend, district=A1} -> {time=Night}*: A1 is downtown, where the bars are.
- *{offense=Verbal Disputes} -> {district=B3}*: verbal disputes are twice as common in B3 (Mattapan).

**Query 3 :** There are about 240 different incidents per day on average. The model error on the last 90 days is 13.6 incidents/day, so about 5.5% of the daily mean, which is pretty good for this kind of data. The components show:
- a weekly pattern: Friday is the busiest day (255/day on average) and Sunday the quietest (214/day)
- a yearly pattern: more incidents in summer (peak in June, about 259/day) and fewer in winter (about 224/day from December to February)
- a holiday effect: Christmas has about 75 incidents less than a normal day
- a trend going slightly up until 2017 and then down in 2018

The busiest day is **1 September 2017** (331 incidents): WHY ? 1 September is "moving day" in Boston, when most leases start.
The quietest days are Christmas 2017 (128), Christmas 2016 (134) and Thanksgiving 2016 (135). The forecast for the next
6 months (September 2018 to February 2019) is around 225 incidents/day, lower than the summer because of the winter effect.

**Query 4 :** With DBSCAN (100 m, 80 points), we find 31 larceny hotspots that hold 37% of all larcenies. The biggest
ones are around Downtown Crossing (Washington St, A1), Boylston St / Copley and Huntington Ave (D4), and State St /
Faneuil Hall (A1). These are all shopping and tourist areas, which matches the D4 rule found in query 2.

**Bonus (weather) :** We joined 1,158 days of NOAA weather data with our daily counts. The correlation between the max
temperature and the number of incidents is **0.44**, which means that there are clearly more incidents on warm days : about 212 per day
when it's between -5 and 0°C, against about 253 per day above 25°C.
Part of this is just the season (summer vs winter), but even inside the same month the correlation stays positive (about 0.24 on average), so warmer days have more incidents than colder days of the same month. Rain has almost no effect (correlation -0.09, about 234 incidents on rainy days vs
241 on dry days). This explains the yearly seasonality found by Prophet in query 3. It's a correlation, not a proof that heat causes crime : people are just more outside when it's warm.

**Conclusion :** Incidents depend a lot on *when* (time of day, weekday, season) and *where* (a few busy districts and
streets). A police department could use the forecast to plan staff by season and day of the week, and the hotspots / rules
to target patrols, for example in shopping streets in the afternoon for larceny and downtown on weekend nights for assaults.

